In [ ]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup

url = "https://banapresso.com/"

driver = webdriver.Chrome()
driver.maximize_window()

driver.get(url)
time.sleep(2)

# 메뉴 이동
action = ActionChains(driver)

first_tag = driver.find_element(
    By.CSS_SELECTOR,
    "#wrap > header > div > ul > li:nth-child(2) > a"
)

second_tag = driver.find_element(
    By.CSS_SELECTOR,
    "#wrap > header > div > ul > li:nth-child(2) > ul > li:nth-child(1) > a > span"
)

action.move_to_element(first_tag) \
      .move_to_element(second_tag) \
      .click() \
      .perform()

# 위치 허용 선택
use_tag = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((
        By.CSS_SELECTOR,
        "#root > div.sc-ff716c63-0.fOAigp > div > div.p_btm"
    ))
)

use_tag.click()

# 매장 목록 로딩 대기
WebDriverWait(driver, 10).until(
    EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, "li.sc-4ba45bc0-0")
    )
)

# 스크롤 끝까지 내리기
while True:
    stores = driver.find_elements(By.CSS_SELECTOR, "li.sc-4ba45bc0-0")
    current_count = len(stores)

    driver.execute_script(
        "arguments[0].scrollIntoView();",
        stores[-1]
    )

    time.sleep(1.5)

    stores = driver.find_elements(By.CSS_SELECTOR, "li.sc-4ba45bc0-0")
    new_count = len(stores)

    # 더 이상 매장 수가 늘어나지 않으면 종료
    if new_count == current_count:
        break

# HTML 가져오기
req = driver.page_source

soup = BeautifulSoup(req, "html.parser")

stores = soup.select("li.sc-4ba45bc0-0")

# 데이터 저장 리스트
store_list = []
addr_list = []
time_list = []
parking_list = []

# 데이터 추출
for store in stores:
    name_tag = store.select_one(".name")
    addr_tag = store.select_one(".address")
    time_tag = store.select_one(".store-time-wrap .time.open p:nth-of-type(2)")
    parking_tag = store.select_one(".store-time-wrap .parking span")

    store_name = name_tag.get_text(strip=True) if name_tag else ""
    store_addr = addr_tag.get_text(strip=True) if addr_tag else ""
    store_time = time_tag.get_text(strip=True) if time_tag else ""
    parking = parking_tag.get_text(strip=True) if parking_tag else ""

    store_list.append(store_name)
    addr_list.append(store_addr)
    time_list.append(store_time)
    parking_list.append(parking)

# 데이터프레임 생성
df = pd.DataFrame({
    'store': store_list,
    'addr': addr_list,
    'time': time_list,
    'parking': parking_list,
})

driver.quit()

print(df)
print("총 매장 수:", len(df))

df.to_csv(
    "banapresso_store.csv",
    index=False,
    encoding="utf-8-sig"
)


          store                            addr         time       parking
0          가락몰점   서울특별시 송파구 양재대로 932, 업무동 1층 로비  07:30~23:30              
1     가산디지털단지역점                서울시 금천구 가산동 60-3  07:00~19:00              
2        가산안양천점   서울 금천구 가산 디지털2로 127-143, 101호  07:00~20:00              
3       가산어반워크점    서울시 금천구 가산디지털2로 135, 1동 142호                           
4     가산우림라이온스점        서울 금천구 가산디지털1로 168 b131호                           
..          ...                             ...          ...           ...
223     회기역사거리점         서울 동대문구 회기로 176 (회기동81)  07:00~22:00              
224       AK금정점  경기도 군포시 금정동 689번지 AK플라자 금정점 2층  07:30~22:30  지하 주차장 이용 가능
225  가산에이스비즈포레점               서울 금천구 가산동 459-23                           
226  가산한라시그마밸리점    서울특별시 금천구 가산디지털2로 53 1층 106호                           
227        명동역점                                                           

[228 rows x 4 columns]
총 매장 수: 228
